# Extraction Probe — de-risking the JD aggregator

Run the extraction approach over a handful of your **real** screenshots and find
where it breaks, *before* the 5-hour sprint. This is verification, not building.

**Before running:** drop ~5–6 JD screenshots into `scratch/screenshots/`, pick the
`.venv` kernel, and set `PROVIDER` below to whichever provider you have credits for.

What to scrutinize:
- Do cropped JDs return clean `null` (not hallucinated company/title)?
- Can vision read logo-only company names?
- Does the seniority ladder give sane labels, with the right `seniority_signal`?
- What leaks in as a bogus "skill" (UI chrome, repeated text)?

In [ ]:
# --- Config -------------------------------------------------------------------
from pathlib import Path

PROVIDER = "anthropic"          # "anthropic" or "openai" — switch to your credited provider
ANTHROPIC_MODEL = "claude-sonnet-4-6"        # strong vision for the read; "claude-haiku-4-5-20251001" is cheaper
OPENAI_MODEL = "gpt-4o"                       # vision-capable; "gpt-4o-mini" is cheaper but weaker on logos
N_TEST = 3                       # how many screenshots to probe this run

SCREENSHOTS_DIR = Path("screenshots")        # relative to scratch/ (where this notebook lives)

from dotenv import load_dotenv
load_dotenv(Path("..") / ".env")             # key lives in the repo-root .env
print(f"Provider: {PROVIDER}")

## Schema — the per-job extraction target
Nullable everywhere: "not stated" must beat a guess.

In [ ]:
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field


class Seniority(str, Enum):
    junior = "Junior"
    mid = "Mid"
    senior = "Senior"


class SeniorityBasis(str, Enum):
    stated = "stated"
    inferred = "inferred"


class Skill(BaseModel):
    raw_text: str = Field(description="the skill exactly as it appeared")
    canonical: str = Field(description="normalized canonical name")


class JobExtraction(BaseModel):
    company: Optional[str] = Field(description="hiring company; null if cropped/not visible")
    title: Optional[str] = Field(description="role name; null if not in frame")
    seniority: Optional[Seniority]
    seniority_signal: Optional[str] = Field(description="the phrase/years the label keyed off")
    seniority_basis: Optional[SeniorityBasis]
    summary: Optional[str] = Field(description="1-2 sentences: what this role wants")
    skills: list[Skill]

## The extraction rules (system prompt)
Encodes the spec: honest nulls, the seniority ladders, canonical map seed, discard UI chrome.

In [ ]:
SYSTEM_PROMPT = """You extract structured data from a SINGLE screenshot of a job posting.
The screenshots are PARTIAL: they may start or end mid-section, and the company/title
may be cropped out or shown only as a logo.

CORE RULE: BE HONEST ABOUT ABSENCE. If a field is not visible in this screenshot,
return null. Never guess or fill gaps. "not stated" beats a wrong guess.

Fields:
- company: the hiring company. May be logo-only (read the logo if you can) or cropped out -> null.
- title: the role name. If the screenshot opens mid-section with no title in frame -> null.
- seniority: one of Junior | Mid | Senior. Usually NOT stated outright -- infer it, but
  follow these ladders STRICTLY (do not freelance):
    Years:    <2yr -> Junior, 2-5yr -> Mid, 5+yr -> Senior
    Language: lead/principal/architect/deep expertise -> Senior;
              proven/production/ownership -> Mid;
              eager to learn/initial experience/strong interest -> Junior
- seniority_signal: the exact phrase or years the label keyed off
  (e.g. "(Junior)", "initial experience or a very strong interest", "5+ years").
- seniority_basis: "stated" if the posting names the level explicitly, else "inferred".
- summary: 1-2 sentences, what this role wants.
- skills: the SET of distinct technical skills this role asks for. For each, give
  raw_text (as it appeared) and canonical (normalized).

NORMALIZATION (canonical) -- collapse variants to ONE canonical name. Seed map (extend sensibly):
- LLMs            <- large language models, LLM, LLM APIs, LLM orchestration
- RAG             <- retrieval-augmented generation
- Agents          <- LLM agents, agentic workflows, multi-agent, agent components
- Prompt engineering <- prompt design/optimization
- Tool calling    <- tool use, function calling, tool/function calling
- CI/CD           <- CI/CD automation
- Keep GCP / AWS / Azure SEPARATE (market signal). Keep LangChain, LlamaIndex, Python, SQL, Docker as-is.

DISCARD UI chrome -- NOT skills: apply buttons, German UI words (Vollzeit), model-name
corner labels (gpt4), verified checkmarks, bookmark/share icons, nav."""

USER_TEXT = "Extract the job posting from this screenshot."

## Extraction — one function per provider, one schema out
Both force structured output against `JobExtraction` so the rest of the notebook is provider-agnostic.

In [ ]:
import base64
import mimetypes


def load_image_b64(path):
    data = Path(path).read_bytes()
    b64 = base64.standard_b64encode(data).decode("utf-8")
    media = mimetypes.guess_type(str(path))[0] or "image/png"
    return b64, media


def _openai_parse(client, **kwargs):
    # .parse graduated from beta in openai v2; fall back to beta for older SDKs.
    try:
        return client.chat.completions.parse(**kwargs)
    except AttributeError:
        return client.beta.chat.completions.parse(**kwargs)


def extract_openai(b64, media):
    from openai import OpenAI
    client = OpenAI()
    completion = _openai_parse(
        client,
        model=OPENAI_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "text", "text": USER_TEXT},
                {"type": "image_url", "image_url": {"url": f"data:{media};base64,{b64}"}},
            ]},
        ],
        response_format=JobExtraction,
    )
    return completion.choices[0].message.parsed


def extract_anthropic(b64, media):
    from anthropic import Anthropic
    client = Anthropic()
    tool = {
        "name": "record_job",
        "description": "Record the extracted job fields. Use null for anything not visible.",
        "input_schema": JobExtraction.model_json_schema(),
    }
    msg = client.messages.create(
        model=ANTHROPIC_MODEL,
        max_tokens=1024,
        system=SYSTEM_PROMPT,
        tools=[tool],
        tool_choice={"type": "tool", "name": "record_job"},
        messages=[{"role": "user", "content": [
            {"type": "image", "source": {"type": "base64", "media_type": media, "data": b64}},
            {"type": "text", "text": USER_TEXT},
        ]}],
    )
    for block in msg.content:
        if block.type == "tool_use":
            return JobExtraction.model_validate(block.input)
    raise RuntimeError("No tool_use block returned")


def extract(path):
    b64, media = load_image_b64(path)
    if PROVIDER == "anthropic":
        return extract_anthropic(b64, media)
    if PROVIDER == "openai":
        return extract_openai(b64, media)
    raise ValueError(f"Unknown PROVIDER: {PROVIDER}")

## Run the probe
Displays each screenshot next to what the model extracted. Eyeball the nulls and the seniority signal.

In [ ]:
from IPython.display import Image, display

exts = {".png", ".jpg", ".jpeg", ".webp"}
shots = sorted(p for p in SCREENSHOTS_DIR.glob("*") if p.suffix.lower() in exts)
print(f"Found {len(shots)} screenshot(s) in {SCREENSHOTS_DIR.resolve()}")
if not shots:
    print("--> Drop some JD screenshots into scratch/screenshots/ and re-run.")

results = []
for p in shots[:N_TEST]:
    print("=" * 70)
    print(p.name)
    display(Image(filename=str(p), width=420))
    try:
        job = extract(p)
        results.append((p.name, job))
        print(job.model_dump_json(indent=2))
    except Exception as e:
        print(f"!! extraction failed: {type(e).__name__}: {e}")

## Aggregate by document frequency
Count **jobs that mention a skill**, not raw mentions — that neutralizes repeated paragraphs and matches the dashboard's chart.

In [ ]:
from collections import Counter

doc_freq = Counter()
for _name, job in results:
    for canon in {s.canonical for s in job.skills}:   # set() -> distinct per job
        doc_freq[canon] += 1

n = len(results)
print(f"Ranked canonical skills across {n} job(s):\n")
for skill, count in doc_freq.most_common():
    print(f"  {count}/{n}  {skill}")

## Notes — where it breaks (fill this in as you go)

- [ ] Cropped JDs return clean `null` for company/title (no hallucination)?
- [ ] Logo-only company names: vision reads them, or null?
- [ ] Seniority labels sane? `seniority_signal` is the *right* evidence?
- [ ] ZDF-style "initial experience / strong interest" -> Junior (calibration case)?
- [ ] Any UI chrome leaking in as a "skill"?
- [ ] Multi-column / skills-in-prose layouts handled?

**Canonical-map gaps found in the real data (extend the seed in the system prompt):**

- ...